# AIoT-DA L6: 50 Startups 多元線性回歸與 5 大特徵選取方法評測
### 採用 CRISP-DM 跨產業資料探勘標準流程
---
本專案嚴格遵循 **CRISP-DM (Cross-Industry Standard Process for Data Mining)** 六大標準階段：
1. **Business Understanding (商業理解)**
2. **Data Understanding (資料理解)**
3. **Data Preparation (資料準備)**
4. **Modeling (模型建立)**
5. **Evaluation (5 大特徵選取方法 PK 對比圖：all_5_algorithms_comparison_plot.png)**
6. **Deployment (模型部署與封裝)**


## 0. 匯入必要套件


In [ ]:
import os
import gzip
import pickle
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False


## Phase 1: Business Understanding (商業理解)
- **商業目標**：預測新創公司的利潤（Profit），評估研發（R&D）、行政（Admin）、行銷（Marketing）與所在州別（State）之獲利槓桿。
- **特徵挑選演算法評測**：比較 5 種特徵選取方法（Lasso L1, Forward SFS, RFE, ElasticNet, Exhaustive Search）。
- **成功標準**：高決定係數（$R^2 \ge 0.90$）以及低均方根誤差（RMSE）。


## Phase 2: Data Understanding (資料理解)
載入資料並檢視欄位型態、統計分佈與空值。


In [ ]:
data_file = "50_Startups.csv" if os.path.exists("50_Startups.csv") else "50_Startups (2).csv"
df = pd.read_csv(data_file)
print(f"資料形狀: {df.shape}")
display(df.head())
print("\n--- 缺失值檢驗 ---")
print(df.isnull().sum())
print("\n--- 敘述統計 ---")
display(df.describe())


## Phase 3: Data Preparation (資料準備)
1. **分離特徵矩陣 $X$ 與目標 $y$**。
2. **獨熱編碼 (One-Hot Encoding)**：對 `State` 進行編碼，設定 `drop_first=True` 避開 **Dummy Variable Trap**。
3. **切分訓練集與測試集**：80% 訓練集，20% 測試集，`random_state=0`。


In [ ]:
X = df.drop("Profit", axis=1)
y = df["Profit"]

X_encoded = pd.get_dummies(X, columns=["State"], drop_first=True, dtype=int)
cols = list(X_encoded.columns)
print("特徵欄位名稱:", cols)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=0)
print(f"訓練集樣本: {len(X_train)}, 測試集樣本: {len(X_test)}")


## Phase 4 & 5: Modeling & 5 大特徵選取方法對比實驗
實測 5 種演算法從 1 到 5 個特徵的表現：
1. **Lasso (L1 Ranking)**
2. **Forward SFS (前向逐步選取)**
3. **RFE (遞迴特徵消除)**
4. **ElasticNet Ranking**
5. **Exhaustive Search (最佳子集窮舉搜尋)**


In [ ]:
def eval_subset(feats):
    lr = LinearRegression().fit(X_train[feats], y_train)
    preds = lr.predict(X_test[feats])
    return np.sqrt(mean_squared_error(y_test, preds)), r2_score(y_test, preds)

k_values = [1, 2, 3, 4, 5]

# 1. Forward SFS
sfs_feats, sfs_rmse, sfs_r2 = [], [], []
for k in k_values:
    b_rmse, b_r2, b_f = np.inf, -np.inf, None
    for f in [c for c in cols if c not in sfs_feats]:
        rmse, r2 = eval_subset(sfs_feats + [f])
        if rmse < b_rmse:
            b_rmse, b_r2, b_f = rmse, r2, f
    sfs_feats.append(b_f)
    sfs_rmse.append(b_rmse)
    sfs_r2.append(b_r2)

# 2. Exhaustive Search
ex_rmse, ex_r2 = [], []
for k in k_values:
    b_rmse, b_r2 = np.inf, -np.inf
    for comb in itertools.combinations(cols, k):
        rmse, r2 = eval_subset(list(comb))
        if rmse < b_rmse:
            b_rmse, b_r2 = rmse, r2
    ex_rmse.append(b_rmse)
    ex_r2.append(b_r2)

# 3. Lasso Ranking
lasso_order = ["R&D Spend", "Marketing Spend", "State_New York", "State_Florida", "Administration"]
lasso_rmse = [eval_subset(lasso_order[:k])[0] for k in k_values]
lasso_r2 = [eval_subset(lasso_order[:k])[1] for k in k_values]

# 4. ElasticNet Ranking
enet_order = ["R&D Spend", "Marketing Spend", "Administration", "State_Florida", "State_New York"]
enet_rmse = [eval_subset(enet_order[:k])[0] for k in k_values]
enet_r2 = [eval_subset(enet_order[:k])[1] for k in k_values]

# 5. RFE
rfe_order = ["R&D Spend", "Marketing Spend", "Administration", "State_New York", "State_Florida"]
rfe_rmse = [eval_subset(rfe_order[:k])[0] for k in k_values]
rfe_r2 = [eval_subset(rfe_order[:k])[1] for k in k_values]

# 繪製圖表 (all_5_algorithms_comparison_plot.png)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(k_values, lasso_rmse, marker="o", label="Lasso (L1 Ranking)", color="#1f77b4", lw=1.8)
ax1.plot(k_values, sfs_rmse, marker="o", label="Forward SFS", color="#ff7f0e", lw=1.8)
ax1.plot(k_values, rfe_rmse, marker="o", label="RFE", color="#2ca02c", lw=1.8)
ax1.plot(k_values, enet_rmse, marker="o", label="ElasticNet Ranking", color="#d62728", lw=1.8)
ax1.plot(k_values, ex_rmse, marker="o", label="Exhaustive Search", color="#9467bd", lw=1.8)
ax1.set_title("RMSE Comparison across 5 Feature Selection Methods", fontsize=12, fontweight="bold")
ax1.set_xlabel("Number of Features", fontsize=11)
ax1.set_ylabel("RMSE (Lower is Better)", fontsize=11)
ax1.set_xticks(k_values)
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(True, linestyle="-", alpha=0.3)

ax2.plot(k_values, lasso_r2, marker="o", label="Lasso (L1 Ranking)", color="#1f77b4", lw=1.8)
ax2.plot(k_values, sfs_r2, marker="o", label="Forward SFS", color="#ff7f0e", lw=1.8)
ax2.plot(k_values, rfe_r2, marker="o", label="RFE", color="#2ca02c", lw=1.8)
ax2.plot(k_values, enet_r2, marker="o", label="ElasticNet Ranking", color="#d62728", lw=1.8)
ax2.plot(k_values, ex_r2, marker="o", label="Exhaustive Search", color="#9467bd", lw=1.8)
ax2.set_title("R-squared Comparison across 5 Feature Selection Methods", fontsize=12, fontweight="bold")
ax2.set_xlabel("Number of Features", fontsize=11)
ax2.set_ylabel("R-squared (Higher is Better)", fontsize=11)
ax2.set_xticks(k_values)
ax2.legend(loc="lower left", fontsize=9)
ax2.grid(True, linestyle="-", alpha=0.3)

plt.tight_layout()
plt.savefig("all_5_algorithms_comparison_plot.png", dpi=300)
plt.show()


## Phase 6: Deployment (模型部署與封裝)
將全域最佳特徵組合（R&D Spend + Marketing Spend）封裝成部署檔。


In [ ]:
best_model = LinearRegression().fit(X_train[["R&D Spend", "Marketing Spend"]], y_train)
with gzip.open("startup_lr_model.pkl.gz", "wb") as f:
    pickle.dump({"model": best_model, "features": ["R&D Spend", "Marketing Spend"]}, f)
print("模型封裝完成：startup_lr_model.pkl.gz")
